In [8]:
import xlwings as xw
import pandas as pd
import numpy as np
import random
from matplotlib import pyplot as plt
from sklearn.preprocessing import PolynomialFeatures,StandardScaler
from sklearn.linear_model import LinearRegression, Lasso,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV,cross_val_score,KFold, cross_val_predict
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF,DotProduct, WhiteKernel
from sklearn.datasets import make_regression
import seaborn as sns
import xgboost as xgb
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'  # 轴标签粗体
plt.rcParams['axes.titleweight'] = 'bold'  # 标题粗体

In [14]:
wb = xw.Book('data.xlsx')
sheet1 = wb.sheets["sheet1"]
features=np.array(sheet1.range('F2:BV2').value)
features_ab=np.array(sheet1.range('F1:BV1').value)
Y=np.array(sheet1.range('C4:C806').value)
X=np.array(sheet1.range('F4:BV806').value)
X=StandardScaler().fit_transform(X)

In [16]:
import warnings
warnings.filterwarnings('ignore')

def n_fold_cv_evaluation(models, X, y, n_folds=10, random_state=42):
    """
    对多个模型进行n-fold交叉验证
    
    参数:
    models: dict, 模型名称和模型实例的字典
    X: 特征数据
    y: 目标值
    n_folds: 折数
    random_state: 随机种子
    """
    
    # 创建K折交叉验证
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    # 存储所有模型的结果
    all_results = {}
    
    print("=" * 100)
    print(f"{'模型名称':<20} {'R²_mean':<12} {'R²_std':<12} {'RMSE_mean':<12} {'RMSE_std':<12}")
    print("=" * 100)
    
    for model_name, model in models.items():
        print(f"\n正在评估模型: {model_name}")
        print("-" * 60)
        
        # 存储每折的结果
        fold_r2_scores = []
        fold_rmse_scores = []
        fold_predictions = []
        fold_actuals = []
        fold_indices = []
        
        fold = 1
        for train_idx, val_idx in kf.split(X):
            # 划分数据
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            # 训练模型
            model.fit(X_train, y_train)
            
            # 预测
            y_pred = model.predict(X_val)
            
            # 计算指标
            r2 = r2_score(y_val, y_pred)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            
            fold_r2_scores.append(r2)
            fold_rmse_scores.append(rmse)
            
            # 存储预测值和实际值
            fold_predictions.extend(y_pred)
            fold_actuals.extend(y_val)
            fold_indices.extend([fold] * len(y_val))
            
            print(f"  Fold {fold}: R² = {r2:.4f}, RMSE = {rmse:.4f}")
            fold += 1
        
        # 计算统计信息
        mean_r2 = np.mean(fold_r2_scores)
        std_r2 = np.std(fold_r2_scores)
        mean_rmse = np.mean(fold_rmse_scores)
        std_rmse = np.std(fold_rmse_scores)
        
        # 创建结果DataFrame
        results_df = pd.DataFrame({
            'Fold': fold_indices,
            'Measured': fold_actuals,
            'Predicted': fold_predictions
        })
        
        all_results[model_name] = {
            'results_df': results_df,
            'r2_scores': fold_r2_scores,
            'rmse_scores': fold_rmse_scores,
            'mean_r2': mean_r2,
            'std_r2': std_r2,
            'mean_rmse': mean_rmse,
            'std_rmse': std_rmse
        }
        
        print(f"\n  {model_name} 汇总:")
        print(f"    R²: {mean_r2:.4f} (+/- {std_r2:.4f})")
        print(f"    RMSE: {mean_rmse:.4f} (+/- {std_rmse:.4f})")
        
        # 打印表格行
        print(f"{model_name:<20} {mean_r2:<12.4f} {std_r2:<12.4f} {mean_rmse:<12.4f} {std_rmse:<12.4f}")
    
    return all_results

def plot_r2_bar_chart(all_results, save_path=None):
    """
    绘制不同模型R²的柱状图（包含误差棒）
    """
    # 设置中文字体
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 用于显示中文
    plt.rcParams['axes.unicode_minus'] = False
    
    # 准备数据
    model_names = list(all_results.keys())
    r2_means = [all_results[name]['mean_r2'] for name in model_names]
    r2_stds = [all_results[name]['std_r2'] for name in model_names]
    
    # 创建图形
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # 设置柱状图位置
    x_pos = np.arange(len(model_names))
    
    # 绘制柱状图（带误差棒）
    bars = ax.bar(x_pos, r2_means, yerr=r2_stds, capsize=8, 
                  color='steelblue', alpha=0.8, edgecolor='black', linewidth=1.5,
                  error_kw={'linewidth': 2, 'ecolor': 'red', 'capsize': 8})
    
    # 添加数值标签
    for i, (bar, mean, std) in enumerate(zip(bars, r2_means, r2_stds)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}\n±{std:.3f}', ha='center', va='bottom', 
                fontsize=9, fontweight='bold')
    
    # 添加水平参考线（R²=0）
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
    
    # 设置y轴范围（留出一些空间给标签）
    y_min = min(0, min(r2_means) - max(r2_stds) - 0.1)
    y_max = max(r2_means) + max(r2_stds) + 0.15
    ax.set_ylim(y_min, y_max)
    
    # 设置标签和标题
    ax.set_xlabel('Models', fontsize=13, fontweight='bold')
    ax.set_ylabel('R² Score', fontsize=13, fontweight='bold')
    ax.set_title('Model Comparison - R² Score (10-fold Cross Validation)', 
                 fontsize=15, fontweight='bold', pad=20)
    
    # 设置x轴刻度
    ax.set_xticks(x_pos)
    ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=11)
    
    # 添加网格
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.set_axisbelow(True)  # 网格在柱状图下方
    
    # 添加背景色
    ax.set_facecolor('#f8f9fa')
    
    # 调整布局
    plt.tight_layout()
    
    # 保存图片
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"\nR²柱状图已保存至: {save_path}")
    
    plt.show()
    
    return fig

def plot_model_comparison(all_results, save_path=None):
    """
    绘制模型比较图（R²和RMSE并排）
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # 子图1: R²分数比较
    ax1 = axes[0]
    model_names = list(all_results.keys())
    r2_means = [all_results[name]['mean_r2'] for name in model_names]
    r2_stds = [all_results[name]['std_r2'] for name in model_names]
    
    x_pos = np.arange(len(model_names))
    bars1 = ax1.bar(x_pos, r2_means, yerr=r2_stds, capsize=5, 
                   color='steelblue', alpha=0.7, edgecolor='black')
    
    # 添加数值标签
    for i, (bar, mean) in enumerate(zip(bars1, r2_means)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax1.set_xlabel('Models', fontsize=12)
    ax1.set_ylabel('R² Score', fontsize=12)
    ax1.set_title('Model Comparison - R² Score (10-fold CV)', fontsize=14, fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(model_names, rotation=45, ha='right')
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.axhline(y=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
    
    # 子图2: RMSE比较
    ax2 = axes[1]
    rmse_means = [all_results[name]['mean_rmse'] for name in model_names]
    rmse_stds = [all_results[name]['std_rmse'] for name in model_names]
    
    bars2 = ax2.bar(x_pos, rmse_means, yerr=rmse_stds, capsize=5,
                   color='coral', alpha=0.7, edgecolor='black')
    
    # 添加数值标签
    for i, (bar, mean) in enumerate(zip(bars2, rmse_means)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax2.set_xlabel('Models', fontsize=12)
    ax2.set_ylabel('RMSE', fontsize=12)
    ax2.set_title('Model Comparison - RMSE (10-fold CV)', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(model_names, rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\n模型比较图已保存至: {save_path}")
    
    plt.show()

def print_summary_table(all_results):
    """
    打印详细的汇总表格
    """
    print("\n" + "=" * 80)
    print("模型性能汇总表 (10-fold Cross Validation)")
    print("=" * 80)
    
    # 创建DataFrame用于显示
    summary_data = []
    for model_name, results in all_results.items():
        summary_data.append({
            'Model': model_name,
            'R² Mean': f"{results['mean_r2']:.4f}",
            'R² Std': f"{results['std_r2']:.4f}",
            'RMSE Mean': f"{results['mean_rmse']:.4f}",
            'RMSE Std': f"{results['std_rmse']:.4f}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # 找出最佳模型
    best_model = max(all_results.items(), key=lambda x: x[1]['mean_r2'])
    print("\n" + "=" * 80)
    print(f"最佳模型: {best_model[0]}")
    print(f"  R²: {best_model[1]['mean_r2']:.4f} (+/- {best_model[1]['std_r2']:.4f})")
    print(f"  RMSE: {best_model[1]['mean_rmse']:.4f} (+/- {best_model[1]['std_rmse']:.4f})")
    print("=" * 80)
    
    return best_model[0]

# 主程序
if __name__ == "__main__":
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
    
    # 定义所有模型
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(
            n_estimators=100, max_depth=40, 
            min_samples_leaf=7, min_samples_split=4, 
            random_state=0
        ),
        'SVR': SVR(kernel="rbf", C=15, gamma=0.01, epsilon=0.05),
        'Lasso': Lasso(alpha=0.1, random_state=42),
        'Ridge': Ridge(alpha=0.5, random_state=42),
        'Gaussian Process': GaussianProcessRegressor(
            kernel=DotProduct() + WhiteKernel(), 
            n_restarts_optimizer=2, 
            random_state=42
        )
    }
    
    # 执行n-fold交叉验证
    print("\n开始10-fold交叉验证评估...")
    print("=" * 100)
    all_results = n_fold_cv_evaluation(models, X, Y, n_folds=10, random_state=42)
    
    # 打印详细汇总表
    best_model_name = print_summary_table(all_results)


开始10-fold交叉验证评估...
模型名称                 R²_mean      R²_std       RMSE_mean    RMSE_std    

正在评估模型: Linear Regression
------------------------------------------------------------
  Fold 1: R² = 0.6264, RMSE = 0.1997
  Fold 2: R² = 0.7692, RMSE = 0.1594
  Fold 3: R² = 0.5249, RMSE = 0.1804
  Fold 4: R² = 0.5635, RMSE = 0.2146
  Fold 5: R² = 0.6256, RMSE = 0.2488
  Fold 6: R² = 0.7321, RMSE = 0.1471
  Fold 7: R² = 0.6710, RMSE = 0.1865
  Fold 8: R² = 0.6358, RMSE = 0.2093
  Fold 9: R² = 0.8532, RMSE = 0.1168
  Fold 10: R² = 0.7469, RMSE = 0.1645

  Linear Regression 汇总:
    R²: 0.6749 (+/- 0.0951)
    RMSE: 0.1827 (+/- 0.0359)
Linear Regression    0.6749       0.0951       0.1827       0.0359      

正在评估模型: Random Forest
------------------------------------------------------------
  Fold 1: R² = 0.6300, RMSE = 0.1987
  Fold 2: R² = 0.8008, RMSE = 0.1480
  Fold 3: R² = 0.6563, RMSE = 0.1534
  Fold 4: R² = 0.6495, RMSE = 0.1923
  Fold 5: R² = 0.7137, RMSE = 0.2176
  Fold 6: R² = 0.7443, 

In [17]:
warnings.filterwarnings('ignore')

def xgboost_nfold_cv_with_predictions(X, y, n_folds=5, params=None, random_state=42):
    """
    XGBoost n-fold交叉验证，记录所有预测值
    """
    
    # 默认XGBoost参数
    if params is None:
        params = {
            'n_estimators': 1000,
            'max_depth': 6,
            'learning_rate': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'early_stopping_rounds': 10,
            'eval_metric': 'rmse',
            'random_state': random_state,
            'verbosity': 0
        }
    
    # 转换为numpy数组
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values
    
    # 创建K折交叉验证
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    # 存储结果
    all_predictions = []
    all_actuals = []
    all_fold_indices = []
    models = []
    
    print(f"开始 {n_folds}-fold 交叉验证...")
    print("=" * 60)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        print(f"\nFold {fold}/{n_folds}")
        print(f"训练集大小: {len(train_idx)}, 验证集大小: {len(val_idx)}")
        
        # 划分数据
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # 训练XGBoost模型
        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        #对val的预测值
        y_pred = model.predict(X_val)
        
        # 创建临时DataFrame来过滤数据
        df = pd.DataFrame({
            'Actual': y_val,
            'Predicted': y_pred
        })
        
        y_pred = df['Predicted'].values
        y_val = df['Actual'].values
        
        all_predictions.extend(y_pred)
        all_actuals.extend(y_val)
        models.append(model)    
              
    # 汇总结果
    results = pd.DataFrame({
        'Measured': all_actuals,
        'Predicted': all_predictions
    })
    
    # 打印汇总统计
    print("\n" + "=" * 60)
    if len(results) > 0:
        print(f"测量值范围: [{results['Measured'].min():.4f}, {results['Measured'].max():.4f}]")
        print(f"预测值范围: [{results['Predicted'].min():.4f}, {results['Predicted'].max():.4f}]")
        
        # 计算整体性能
        r2 = np.corrcoef(results['Measured'], results['Predicted'])[0, 1] ** 2
        rmse = np.sqrt(mean_squared_error(results['Measured'], results['Predicted']))
        mae = mean_absolute_error(results['Measured'], results['Predicted'])
        
        print(f"整体 R²: {r2:.4f}")
        print(f"整体 RMSE: {rmse:.4f}")
        print(f"整体 MAE: {mae:.4f}")
    
    return results, models

def plot_predicted_vs_measured(results, save_path=None):
    """
    绘制Predicted vs Measured散点图
    """
    
    # 设置中文字体（如果需要）
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 用于显示中文
    plt.rcParams['axes.unicode_minus'] = False
    
    y_train = results['Measured']
    y_pred_train = results['Predicted']
    # 定义数值范围
    lower_bound_train = 1  # 下限
    upper_bound_train = 2.1  # 上限
    lower_bound_pred = 0.95  # 下限

    # 创建布尔掩码：保留y_train在范围内的数据
    mask = (y_train >= lower_bound_train) & (y_train <= upper_bound_train)&(y_pred_train >= lower_bound_pred)

    # 同时过滤两个数组
    y_train = y_train[mask]
    y_pred_train = y_pred_train[mask]

    # 创建图形
    fig, ax = plt.subplots(figsize=(7, 7))
    
    # 绘制散点图
    ax.scatter(y_train, y_pred_train, 
               alpha=0.6, s=70, color='green', edgecolors='black', linewidth=0.5)
    
    # 添加对角线（完美预测线）
    min_val = min(y_train.min(), y_pred_train.min())
    max_val = max(y_train.max(), y_pred_train.max())
    # 扩展一点边距
    '''margin = (max_val - min_val) * 0.05
    min_val -= margin
    max_val += margin'''
    
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', 
            linewidth=2, label='Perfect Prediction', alpha=0.8)
    
    # 计算统计信息
    r2 = np.corrcoef(results['Measured'], results['Predicted'])[0, 1] ** 2
    rmse = np.sqrt(np.mean((results['Measured'] - results['Predicted']) ** 2))
    mae = np.mean(np.abs(results['Measured'] - results['Predicted']))
    
    # 设置标签和标题
    ax.set_xlabel(r'Measured $\mathit{Δd}$ (Å)', fontsize=28)
    ax.set_ylabel(r'Predicted $\mathit{Δd}$ (Å)', fontsize=28)
    plt.xticks(np.arange(1, 2.1,0.3),fontsize=24)
    plt.yticks(np.arange(1, 2.1,0.3),fontsize=24)
    # 添加统计信息文本框
    text = (
            f'R² = {r2:.2f}\n'
            f'RMSE = {rmse:.2f}Å\n'
            f'MAE = {mae:.2f}Å\n')
    
    ax.text(0.95, 0.00,text, transform=ax.transAxes, 
            fontsize=24,verticalalignment='bottom',
         horizontalalignment='right')
    
    #ax.legend(loc='best', fontsize=24)
    ax.axis('equal')
    ax.spines['bottom'].set_linewidth(3)  
    ax.spines['left'].set_linewidth(3)  
    ax.spines['bottom'].set_color('black')  
    ax.spines['left'].set_color('black')
    ax.spines['top'].set_linewidth(3)  
    ax.spines['right'].set_linewidth(3)
    ax.spines['top'].set_color('black') 
    ax.spines['right'].set_color('black') 
    ax.spines['top'].set_visible(True)  
    ax.spines['right'].set_visible(True)  
    plt.tight_layout()
    
    # 保存图片
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\n图片已保存至: {save_path}")
    
    plt.show()
    
    return fig

# 使用示例
if __name__ == "__main__":
    # 设置XGBoost参数
    xgb_params = {
        'n_estimators': 100,
        'max_depth': 5,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'early_stopping_rounds': 10,
        'eval_metric': 'rmse',
        'random_state': 42,
        'verbosity': 0,
        'min_child_weight':3,
        'gamma':0.05
    }
    
    # 执行交叉验证（自动过滤数据）
    results, models = xgboost_nfold_cv_with_predictions(
        X, Y, 
        n_folds=10, 
        params=xgb_params,
        random_state=42
    )

        # 绘制散点图
    #plot_predicted_vs_measured(results)
    #plot_predicted_vs_measured(results, save_path='XGBoost.png')
        # 保存预测结果到CSV
    '''results.to_csv('cross_validation_predictions.csv', index=False)
        print("\n预测结果已保存至: cross_validation_predictions.csv")
    else:
        print("\n警告: 没有符合过滤条件的数据!")'''

开始 10-fold 交叉验证...

Fold 1/10
训练集大小: 722, 验证集大小: 81

Fold 2/10
训练集大小: 722, 验证集大小: 81

Fold 3/10
训练集大小: 722, 验证集大小: 81

Fold 4/10
训练集大小: 723, 验证集大小: 80

Fold 5/10
训练集大小: 723, 验证集大小: 80

Fold 6/10
训练集大小: 723, 验证集大小: 80

Fold 7/10
训练集大小: 723, 验证集大小: 80

Fold 8/10
训练集大小: 723, 验证集大小: 80

Fold 9/10
训练集大小: 723, 验证集大小: 80

Fold 10/10
训练集大小: 723, 验证集大小: 80

测量值范围: [0.0026, 3.0561]
预测值范围: [0.4766, 2.0958]
整体 R²: 0.7506
整体 RMSE: 0.1645
整体 MAE: 0.0896
